# Food Safety Analysis with biotech-ml-toolkit

This notebook demonstrates the 7 food safety models:
1. NutriScorePredictor -- Nutri-Score A-E grading
2. AllergenNER -- Big 9 allergen detection in text
3. AdditiveRiskScorer -- E-number risk scoring
4. HACCPClassifier -- HACCP hazard text classification
5. IngredientNER -- Ingredient text parsing
6. NutritionalAnomalyDetector -- Nutrient anomaly detection
7. ProductLookup -- OpenFoodFacts barcode lookup

## 1. Nutri-Score (rule-based fallback, no artifacts needed)

In [ ]:
from biotech_ml.food.nutriscore_predictor import NutriScorePredictor

model = NutriScorePredictor()
model._loaded = True  # rule-based fallback, no artifact needed

# Healthy food
result = model.predict({
    "energy_kcal": 50, "fat": 0.5, "saturated_fat": 0.1,
    "sugar": 3.0, "salt": 0.01, "protein": 5.0, "fiber": 4.0,
})
print(f"Healthy food: Grade={result['grade']}, Score={result['score']}")

# Junk food
result = model.predict({
    "energy_kcal": 550, "fat": 30.0, "saturated_fat": 15.0,
    "sugar": 35.0, "salt": 3.0, "protein": 5.0, "fiber": 1.0,
})
print(f"Junk food:    Grade={result['grade']}, Score={result['score']}")

## 2. Allergen NER (rule-based, no artifacts needed)

In [ ]:
from biotech_ml.food.allergen_ner import AllergenNER, _build_compiled_patterns

ner = AllergenNER()
ner._compiled_patterns = _build_compiled_patterns()
ner._loaded = True

text = "Ingredients: wheat flour, sugar, butter, egg yolk, soy lecithin, almond extract, sesame seeds"
result = ner.predict({"ingredient_text": text})

print(f"Found {len(result['allergens'])} allergens in: {text[:60]}...")
for a in result["allergens"]:
    print(f"  [{a['span_start']}:{a['span_end']}] {a['name']} -> {a['category']}")

## 3. Additive Risk Scoring (knowledge-base fallback)

In [ ]:
from biotech_ml.food.additive_risk import AdditiveRiskScorer, ADDITIVE_RISK_DB

scorer = AdditiveRiskScorer()
scorer._known_additives = sorted(ADDITIVE_RISK_DB.keys())
scorer._loaded = True

# Safe additives
result = scorer.predict({"additives": ["E330", "E412", "E500"]})
print(f"Safe combo: risk={result['risk_score']}")

# Risky additives
result = scorer.predict({"additives": ["E171", "E250", "E320"]})
print(f"Risky combo: risk={result['risk_score']}")
for a in result["anomalies"]:
    print(f"  {a['additive']}: {a['risk_level']} - {a['reason']}")

## 4. HACCP Hazard Classification (keyword fallback)

In [ ]:
from biotech_ml.food.haccp_classifier import HACCPClassifier

clf = HACCPClassifier()
clf._loaded = True  # keyword fallback

texts = [
    "Salmonella detected in raw chicken sample during inspection",
    "Pesticide residue exceeds limits in strawberry batch",
    "Metal fragment found in cereal production line",
    "All temperature records within normal HACCP limits",
]

for text in texts:
    result = clf.predict({"text": text})
    status = "SAFE" if result["is_safe"] else result["category"].upper()
    print(f"  [{status}] (conf={result['confidence']:.2f}) {text[:55]}...")

## 5. Ingredient Parsing (rule-based)

In [ ]:
from biotech_ml.food.ingredient_ner import IngredientNER

parser = IngredientNER()
parser._loaded = True

result = parser.predict({"text": "200g flour, 100ml milk (3.5%), 2 eggs, 50g sugar, 5g salt"})
print("Parsed ingredients:")
for ing in result["ingredients"]:
    parts = [ing['name']]
    if ing['quantity']:
        parts.append(f"{ing['quantity']}{ing['unit'] or ''}")
    if ing['percentage']:
        parts.append(f"{ing['percentage']}%")
    print(f"  {' | '.join(parts)}")